# 01 · Data preparation (M2, Data Engineer)

Builds the working dataset from CDC BRFSS 2023 and documents provenance.
Run top to bottom. Heavy steps (reading the 1.2 GB XPT) are skipped when the output already exists; set `REBUILD = True` to force them.

**Inputs:** `data/raw/LLCP2023.XPT` (see `data/README.md`) · **Output:** `data/processed/brfss2023_working.parquet`
**Rule:** values stay as raw CDC codes. Cleaning and recoding belong to M3.

In [ ]:
import subprocess, sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
XPT = ROOT / "data/raw/LLCP2023.XPT"
OUT = ROOT / "data/processed/brfss2023_working.parquet"
REBUILD = False   # True = re-run steps 1 and 2 even if outputs exist

print("python", sys.version.split()[0], "| pandas", pd.__version__)
print("root:", ROOT)
assert XPT.exists(), f"Missing {XPT}. Download and unzip LLCP2023XPT.zip as described in data/README.md"


## Step 1 · Verify the source file against the CDC codebook

In [ ]:
VERIFY_JSON = ROOT / "docs/source_verification.json"
if REBUILD or not VERIFY_JSON.exists():
    subprocess.run([sys.executable, str(ROOT / "scripts/01_verify_source.py"),
                    "--xpt", str(XPT), "--docs", str(ROOT / "docs/cdc"),
                    "--out", str(VERIFY_JSON)], check=True, cwd=ROOT)
v = json.loads(VERIFY_JSON.read_text(encoding="utf-8"))
print("all_checks_passed:", v["all_checks_passed"])
print("XPT sha256:", v["files"]["xpt"]["sha256"])
print("columns in file:", v["n_columns"]["observed"], "| rows:", v["checks"]["rows"]["observed"])
assert v["all_checks_passed"]


## Step 2 · Build the working dataset (29 columns, raw codes)

In [ ]:
if REBUILD or not OUT.exists():
    subprocess.run([sys.executable, str(ROOT / "scripts/02_build_working_dataset.py"),
                    "--xpt", str(XPT), "--out", str(OUT), "--docs", str(ROOT / "docs")],
                   check=True, cwd=ROOT)
df = pd.read_parquet(OUT)
print(df.shape)
df.head()


## Step 3 · Quick look (no cleaning)

In [ ]:
prof = pd.read_csv(ROOT / "docs/data_profile.csv")
prof[["variable", "dtype", "n_missing", "pct_missing", "n_unique", "min", "max"]]


In [ ]:
# Outcome components: raw codes, 1 = Yes, 2 = No, 7 = Don't know, 9 = Refused, <NA> = blank
for c in ["CVDINFR4", "CVDCRHD4", "CVDSTRK3", "_MICHD"]:
    print(c, df[c].value_counts(dropna=False).sort_index().to_dict())


In [ ]:
# Optional: a 1,000-row sample you can open in Excel (fixed seed). For viewing only; not for analysis.
SAMPLE = ROOT / "data/processed/sample_1000.csv"
df.sample(1000, random_state=42).to_csv(SAMPLE, index=False)
print("wrote", SAMPLE)


## Hand-off notes
* Variable meanings and codebook counts: `docs/variable_dictionary.md`.
* `_MICHD` excludes stroke. Build `CVD_ANY` from `CVDINFR4`, `CVDCRHD4`, `CVDSTRK3` (M3).
* Use `_LLCPWT` with `_STSTR` and `_PSU` for weighted estimates (M3).
* Open decisions are listed in `data/README.md`, section 6.